# WP2 · Nettoyage des données BAAC 2019-2024
**Tâches WP2-07 (règles de nettoyage) et WP2-08 (implémentation et exécution)**

## Objectif
Produire, à partir des quatre fichiers Silver harmonisés du dépôt, une version nettoyée des données (`data/clean/`), accompagnée d'un registre des règles, d'un journal chiffré des transformations et de contrôles d'intégrité.

## Principes appliqués
| Principe | Justification |
|---|---|
| Aucune ligne supprimée | Les anomalies sont signalées par des colonnes `flag_*` (0/1). Le WP3 choisit d'exclure ou non. |
| Codes catégoriels conservés (0, -1) | Dans la BAAC, `0` signifie selon la variable « Aucun », « Sans objet » ou « Inconnu », et `-1` « Non renseigné ». Ce sont des modalités valides des dimensions. |
| Mesures numériques neutralisées | Un `-1` dans `vma`, `nbv` ou l'âge fausserait une moyenne : il devient vide dans une colonne `*_num`, l'original étant conservé. |
| Silver jamais modifié | Le nettoyage lit `data/silver/` et écrit dans `data/clean/`. |
| Traçabilité | Chaque règle a un identifiant (registre) et un compteur de valeurs touchées par année (journal). |
| Contrôle avant écriture | Les fichiers propres ne sont écrits que si tous les contrôles d'intégrité réussissent. |

## 1. Récupération des données Silver
Les données brutes ont déjà été collectées et harmonisées par le WP2 (WP2-02 et WP2-06). On clone le dépôt et on télécharge uniquement les fichiers Silver, stockés avec Git LFS.

In [1]:
!apt-get -qq install -y git-lfs > /dev/null
!git lfs install
![ -d P2-BAAC-2019-2024 ] || git clone -q https://github.com/Moha-Reda/P2-BAAC-2019-2024.git
%cd P2-BAAC-2019-2024
!git lfs pull --include="data/silver/*"
!ls -lh data/silver

Git LFS initialized.
/content/P2-BAAC-2019-2024
total 137M
-rw-r--r-- 1 root root 31M Nov  5 23:55 caracteristiques_2019_2024.csv
-rw-r--r-- 1 root root 24M Nov  5 23:55 lieux_2019_2024.csv
-rw-r--r-- 1 root root 55M Nov  5 23:56 usagers_2019_2024.csv
-rw-r--r-- 1 root root 28M Nov  5 23:55 vehicules_2019_2024.csv


## 2. Registre des règles (WP2-07)
Le registre reprend les règles R001 à R008 définies lors du profilage et les complète avec R009 à R019. Les règles R016 à R018 comportent des seuils à faire valider par le responsable WP2.

In [2]:
%%writefile metadata/nettoyage/registre_regles.csv
id;famille;variable;annees;probleme;action_proposee;statut
R001;caracteristiques;Accident_Id;2022;Nom different;Renommer en Num_Acc dans Silver;VALIDEE
R002;usagers;id_usager;2019-2020;Colonne absente;Conserver vide sans inventer identifiant;VALIDEE
R003;lieux;larrout;2019-2024;-1 tres frequent (2020-2024), vide (2019);Colonne larrout_num : -1 et non numerique -> vide / original conserve;PROPOSEE
R004;lieux;v2;2019-2024;Marqueurs N/A null;Traite par R010 (marqueurs -> vide);PROPOSEE
R005;lieux;lartpc;2019-2024;Tres forte incompletude;Conservee / colonne lartpc_num / a deconseiller en analyse;PROPOSEE
R006;vehicules;occutc;2019-2024;Tres forte incompletude;Conservee / occutc_num / ne concerne que les transports en commun;PROPOSEE
R007;usagers;toutes;2019-2020;164 doublons exacts supplementaires;Conserves + flag_doublon_exact (1 = repetition) / WP3 decide de les exclure ou non;PROPOSEE
R008;lieux;Num_Acc;2023-2024;Plusieurs lignes par accident;Conservees + nb_lieux_accident et rang_lieu (rang 1 = lieu principal pour jointure 1-1);PROPOSEE
R009;toutes;toutes;2019-2024;Espaces parasites;Suppression des espaces en debut/fin;PROPOSEE
R010;toutes;toutes;2019-2024;Marqueurs textuels d absence (N/A null #ERREUR);Remplaces par vide;PROPOSEE
R011;toutes;variables categorielles;2019-2024;Codes 0 et -1 de sens variable;CONSERVES tels quels (0 = Aucun/Sans objet/Inconnu selon variable) / libelles via modalites_baac_finalisees.csv;PROPOSEE
R012;caracteristiques;dep com;2019-2024;Codes sans zeros a gauche possibles;dep sur 2 caracteres, com sur 5 (2A/2B et DOM inchanges);PROPOSEE
R013;caracteristiques;hrmn;2019-2024;Format heure;Normalise HH:MM + colonne heure + flag_heure_invalide;PROPOSEE
R014;caracteristiques;jour mois an;2019-2024;Date eclatee;Colonnes date_accident (AAAA-MM-JJ) et jour_semaine (1=lundi) + flag_date_invalide;PROPOSEE
R015;caracteristiques;lat long;2019-2024;Coordonnees (0,0) ou hors France;(0,0) -> vide / hors metropole/DROM-COM -> flag_coord_hors_territoire (conservees);PROPOSEE
R016;lieux;nbv;2019-2024;-1, #ERREUR, valeurs extremes;nbv_num : vide si -1 ou >12 + flag;A_VALIDER_CHEF
R017;lieux;vma;2019-2024;Vitesses impossibles (ex. 500, 700);vma_num : vide hors [5-130] + flag;A_VALIDER_CHEF
R018;usagers;an_nais;2019-2024;Annees de naissance aberrantes;Colonne age = annee - an_nais / vide hors [0-110] + flag;A_VALIDER_CHEF
R019;usagers;grav;2019-2024;Indicateurs de gravite;Colonnes est_tue, est_blesse_hospitalise, est_blesse_leger, est_indemne (hospitalises : non comparable avant 2018, indicateur non labellise depuis 2019);PROPOSEE


Overwriting metadata/nettoyage/registre_regles.csv


In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.read_csv('metadata/nettoyage/registre_regles.csv', sep=';', encoding='utf-8-sig')

,id,famille,variable,annees,probleme,action_proposee,statut
0,R001,caracteristiques,Accident_Id,2022,Nom different,Renommer en Num_Acc dans Silver,VALIDEE
1,R002,usagers,id_usager,2019-2020,Colonne absente,Conserver vide sans inventer identifiant,VALIDEE
2,R003,lieux,larrout,2019-2024,"-1 tres frequent (2020-2024), vide (2019)",Colonne larrout_num : -1 et non numerique -> vide / original conserve,PROPOSEE
3,R004,lieux,v2,2019-2024,Marqueurs N/A null,Traite par R010 (marqueurs -> vide),PROPOSEE
4,R005,lieux,lartpc,2019-2024,Tres forte incompletude,Conservee / colonne lartpc_num / a deconseiller en analyse,PROPOSEE
5,R006,vehicules,occutc,2019-2024,Tres forte incompletude,Conservee / occutc_num / ne concerne que les transports en commun,PROPOSEE
6,R007,usagers,toutes,2019-2020,164 doublons exacts supplementaires,Conserves + flag_doublon_exact (1 = repetition) / WP3 decide de les exclure ou non,PROPOSEE
7,R008,lieux,Num_Acc,2023-2024,Plusieurs lignes par accident,Conservees + nb_lieux_accident et rang_lieu (rang 1 = lieu principal pour jointure 1-1),PROPOSEE
8,R009,toutes,toutes,2019-2024,Espaces parasites,Suppression des espaces en debut/fin,PROPOSEE
9,R010,toutes,toutes,2019-2024,Marqueurs textuels d absence (N/A null #ERREUR),Remplaces par vide,PROPOSEE


## 3. Script de nettoyage (WP2-08)
Le script est enregistré dans `scripts/` pour pouvoir être relancé par n'importe quel membre du projet avec `python scripts/nettoyer_baac.py`.

| Famille | Traitements |
|---|---|
| Toutes | Espaces superflus (R009), marqueurs `N/A`, `null`, `#ERREUR` → vide (R010) |
| Caractéristiques | `dep`/`com` complétés (R012), heure HH:MM + `heure` (R013), `date_accident` et `jour_semaine` (R014), coordonnées (0,0) et hors territoire (R015) |
| Lieux | `larrout_num`, `lartpc_num` (R003, R005), `nbv_num` (R016), `vma_num` (R017), lieux multiples : `nb_lieux_accident`, `rang_lieu` (R008) |
| Véhicules | `occutc_num` (R006) |
| Usagers | Doublons exacts signalés (R007), `age` (R018), indicateurs de gravité (R019) |

In [4]:
%%writefile scripts/nettoyer_baac.py
"""
WP2-08 - NETTOYAGE BAAC 2019-2024
Tache WP2-08 (regles WP2-07)

Entree  : data/silver/{famille}_2019_2024.csv   (jamais modifies)
Sorties : data/clean/{famille}_2019_2024.csv     (CSV ; UTF-8 BOM)
          metadata/nettoyage/journal_transformations.csv
          metadata/nettoyage/controles_apres_nettoyage.csv

Principes (note de cadrage, section 9.3) :
- aucune ligne supprimee : les anomalies sont signalees par des colonnes flag_*
- les codes categoriels (0, -1...) sont CONSERVES : leur sens depend de la
  variable (0 = "Aucun", "Sans objet", "Inconnu"...). Ils deviennent des
  modalites "Non renseigne" dans les dimensions du WP3.
- seules les variables NUMERIQUES (mesures) voient leurs marqueurs
  d'absence (-1, N/A, #ERREUR...) remplaces par vide, car un -1 fausserait
  une moyenne ou une somme.
- les colonnes d'origine ne sont pas reecrites quand une valeur est
  invalidee : une colonne derivee est creee a cote.
- chaque regle est journalisee avec le nombre de valeurs touchees par annee.

Lancement : python scripts/nettoyer_baac.py
"""

from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
SILVER = ROOT / "data" / "silver"
CLEAN = ROOT / "data" / "clean"
META = ROOT / "metadata" / "nettoyage"

FAMILLES = ["caracteristiques", "lieux", "vehicules", "usagers"]

# Marqueurs textuels d'absence observes dans le profilage (v2 : N/A, null ;
# nbv : #ERREUR dans certains millesimes)
MARQUEURS_VIDES = ["N/A", "n/a", "NA", "NULL", "null", "nan", "NaN", "#ERREUR", "#VALEURMULTI"]

# Parametres metier (a valider par le chef WP2, regle R017/R018)
VMA_MIN, VMA_MAX = 5, 130
AGE_MIN, AGE_MAX = 0, 110
NBV_MAX = 12

# Emprise approximative France metropolitaine + DROM-COM
ZONES_GEO = [
    # (lat_min, lat_max, long_min, long_max, nom)
    (41.0, 51.5, -5.5, 10.0, "metropole"),
    (15.5, 18.5, -63.5, -60.5, "antilles"),
    (1.5, 6.5, -55.0, -51.0, "guyane"),
    (-21.6, -20.7, 55.0, 56.0, "reunion"),
    (-13.2, -12.4, 44.8, 45.5, "mayotte"),
    (46.5, 47.5, -57.0, -55.9, "saint_pierre"),
    (-23.0, -9.0, -155.0, -134.0, "polynesie"),
    (-23.0, -19.0, 163.0, 169.0, "nouvelle_caledonie"),
    (-14.5, -13.0, -178.5, -175.5, "wallis_futuna"),
]

journal = []


def log(regle, famille, serie_masque, df, description):
    """Enregistre le nombre de valeurs touchees par annee pour une regle."""
    if serie_masque is None:
        return
    compte = serie_masque.groupby(df["annee"]).sum()
    for annee, n in compte.items():
        journal.append({
            "regle": regle, "famille": famille, "annee": annee,
            "valeurs_touchees": int(n), "description": description,
        })


def lire(famille):
    chemin = SILVER / f"{famille}_2019_2024.csv"
    if not chemin.exists():
        sys.exit(f"Fichier introuvable : {chemin} (as-tu fait 'git lfs pull' ?)")
    df = pd.read_csv(chemin, sep=";", dtype=str, encoding="utf-8-sig",
                     keep_default_na=False)
    if df.iloc[0, 0].startswith("version https://git-lfs"):
        sys.exit(f"{chemin} est un pointeur Git LFS : lance 'git lfs pull'.")
    return df


def ecrire(df, famille):
    CLEAN.mkdir(parents=True, exist_ok=True)
    df.to_csv(CLEAN / f"{famille}_2019_2024.csv", sep=";", index=False,
              encoding="utf-8-sig")


# --------------------------------------------------------------------------
# Regles communes
# --------------------------------------------------------------------------

def r009_espaces(df, famille):
    """R009 : suppression des espaces en debut/fin de valeur."""
    total = pd.Series(False, index=df.index)
    for c in df.columns:
        nettoye = df[c].str.strip()
        total |= nettoye != df[c]
        df[c] = nettoye
    log("R009", famille, total, df, "Espaces superflus supprimes")
    return df


def r010_marqueurs_vides(df, famille):
    """R010 : marqueurs textuels d'absence -> vide (toutes colonnes)."""
    total = pd.Series(False, index=df.index)
    for c in df.columns:
        m = df[c].isin(MARQUEURS_VIDES)
        total |= m
        df.loc[m, c] = ""
    log("R010", famille, total, df, "Marqueurs N/A, null, #ERREUR remplaces par vide")
    return df


def numerique(df, famille, col, regle, nouvelle_col=None, mini=None, maxi=None):
    """Convertit une mesure en nombre : -1 et non-numeriques -> vide,
    hors bornes -> vide + flag. L'original reste dans `col` si nouvelle_col."""
    cible = nouvelle_col or col
    brut = df[col].str.replace(",", ".", regex=False)
    val = pd.to_numeric(brut, errors="coerce")
    non_renseigne = (val == -1) | (val.isna() & (brut != ""))
    val[val == -1] = np.nan
    hors = pd.Series(False, index=df.index)
    if mini is not None:
        hors |= val < mini
    if maxi is not None:
        hors |= val > maxi
    val[hors] = np.nan
    df[cible] = val.map(lambda x: "" if pd.isna(x) else (str(int(x)) if float(x).is_integer() else str(x)))
    log(regle, famille, non_renseigne, df, f"{col} : -1 ou non numerique -> vide")
    if mini is not None or maxi is not None:
        df[f"flag_{cible}_hors_bornes"] = hors.astype(int).astype(str)
        log(regle, famille, hors, df, f"{col} hors [{mini}, {maxi}] -> vide + flag")
    return df


# --------------------------------------------------------------------------
# Caracteristiques
# --------------------------------------------------------------------------

def nettoyer_caracteristiques(df):
    f = "caracteristiques"

    # R012 : dep sur 2 caracteres (01..95, 2A, 2B, 971...), com sur 5
    dep_avant = df["dep"].copy()
    est_chiffre = df["dep"].str.fullmatch(r"\d")
    df.loc[est_chiffre, "dep"] = "0" + df.loc[est_chiffre, "dep"]
    com_court = df["com"].str.fullmatch(r"\d{1,4}")
    df.loc[com_court, "com"] = df.loc[com_court, "com"].str.zfill(5)
    log("R012", f, (dep_avant != df["dep"]) | com_court, df,
        "Codes dep/com completes par des zeros a gauche")

    # R013 : heure HH:MM + colonne heure
    h = df["hrmn"].str.replace(":", "", regex=False).str.zfill(4)
    valide = h.str.fullmatch(r"([01]\d|2[0-3])[0-5]\d")
    modifie = valide & ((h.str[:2] + ":" + h.str[2:]) != df["hrmn"])
    df.loc[valide, "hrmn"] = h[valide].str[:2] + ":" + h[valide].str[2:]
    df["heure"] = np.where(valide, h.str[:2].astype(str).str.lstrip("0").replace("", "0"), "")
    df["flag_heure_invalide"] = (~valide).astype(int).astype(str)
    log("R013", f, modifie, df, "hrmn normalise au format HH:MM")
    log("R013", f, ~valide, df, "hrmn invalide -> flag (heure vide)")

    # R014 : date complete de l'accident (AAAA-MM-JJ), controle annee == an
    date = pd.to_datetime(
        df["an"].str.zfill(4) + "-" + df["mois"].str.zfill(2) + "-" + df["jour"].str.zfill(2),
        format="%Y-%m-%d", errors="coerce")
    df["date_accident"] = date.dt.strftime("%Y-%m-%d").fillna("")
    df["jour_semaine"] = date.dt.dayofweek.add(1).astype("Int64").astype(str).replace("<NA>", "")
    incoherent = date.isna() | (df["an"] != df["annee"])
    df["flag_date_invalide"] = incoherent.astype(int).astype(str)
    log("R014", f, incoherent, df, "Date invalide ou an different du millesime -> flag")

    # R015 : coordonnees numeriques, (0,0) -> vide, hors territoire -> flag
    lat = pd.to_numeric(df["lat"], errors="coerce")
    lon = pd.to_numeric(df["long"], errors="coerce")
    zero = (lat == 0) & (lon == 0)
    lat[zero] = np.nan
    lon[zero] = np.nan
    dans_zone = pd.Series(False, index=df.index)
    for la1, la2, lo1, lo2, _ in ZONES_GEO:
        dans_zone |= lat.between(la1, la2) & lon.between(lo1, lo2)
    hors = lat.notna() & lon.notna() & ~dans_zone
    df["lat"] = lat.map(lambda x: "" if pd.isna(x) else repr(float(x)))
    df["long"] = lon.map(lambda x: "" if pd.isna(x) else repr(float(x)))
    df["flag_coord_hors_territoire"] = hors.astype(int).astype(str)
    log("R015", f, zero, df, "Coordonnees (0,0) -> vide")
    log("R015", f, hors, df, "Coordonnees hors territoire francais -> flag (conservees)")
    return df


# --------------------------------------------------------------------------
# Lieux
# --------------------------------------------------------------------------

def nettoyer_lieux(df):
    f = "lieux"
    # R003 / R016 : largeurs et nombre de voies = mesures
    df = numerique(df, f, "larrout", "R003", "larrout_num")
    df = numerique(df, f, "lartpc", "R005", "lartpc_num")
    df = numerique(df, f, "nbv", "R016", "nbv_num", mini=0, maxi=NBV_MAX)
    # R017 : vitesse maximale autorisee
    df = numerique(df, f, "vma", "R017", "vma_num", mini=VMA_MIN, maxi=VMA_MAX)

    # R008 : plusieurs lieux par accident (2023-2024) -> conserves + compteur
    df["nb_lieux_accident"] = df.groupby("Num_Acc")["Num_Acc"].transform("size").astype(str)
    df["rang_lieu"] = (df.groupby("Num_Acc").cumcount() + 1).astype(str)
    log("R008", f, df["rang_lieu"] != "1", df,
        "Lignes de lieux supplementaires conservees (rang_lieu > 1)")
    return df


# --------------------------------------------------------------------------
# Vehicules
# --------------------------------------------------------------------------

def nettoyer_vehicules(df):
    f = "vehicules"
    # R006 : occupants transport en commun = mesure, vide si non applicable
    df = numerique(df, f, "occutc", "R006", "occutc_num")
    return df


# --------------------------------------------------------------------------
# Usagers
# --------------------------------------------------------------------------

def nettoyer_usagers(df):
    f = "usagers"

    # R007 : doublons exacts signales, jamais supprimes
    cols = [c for c in df.columns if c != "annee"]
    doublon = df.duplicated(subset=cols + ["annee"], keep="first")
    df["flag_doublon_exact"] = doublon.astype(int).astype(str)
    log("R007", f, doublon, df, "Doublon exact signale (conserve)")

    # R018 : age a la date de l'accident
    an_nais = pd.to_numeric(df["an_nais"], errors="coerce")
    age = pd.to_numeric(df["annee"]) - an_nais
    hors = age.notna() & ((age < AGE_MIN) | (age > AGE_MAX))
    age[hors] = np.nan
    df["age"] = age.map(lambda x: "" if pd.isna(x) else str(int(x)))
    df["flag_age_hors_bornes"] = hors.astype(int).astype(str)
    log("R018", f, hors, df, f"Age hors [{AGE_MIN}, {AGE_MAX}] -> vide + flag")

    # R019 : indicateurs de gravite (1 indemne, 2 tue, 3 hospitalise, 4 leger)
    df["est_tue"] = (df["grav"] == "2").astype(int).astype(str)
    df["est_blesse_hospitalise"] = (df["grav"] == "3").astype(int).astype(str)
    df["est_blesse_leger"] = (df["grav"] == "4").astype(int).astype(str)
    df["est_indemne"] = (df["grav"] == "1").astype(int).astype(str)
    return df


TRAITEMENTS = {
    "caracteristiques": nettoyer_caracteristiques,
    "lieux": nettoyer_lieux,
    "vehicules": nettoyer_vehicules,
    "usagers": nettoyer_usagers,
}


# --------------------------------------------------------------------------
# Controles apres nettoyage
# --------------------------------------------------------------------------

def controler(tables, volumes_avant):
    res = []

    def ajoute(nom, ok, detail=""):
        res.append({"controle": nom, "resultat": "OK" if ok else "ECHEC", "detail": detail})

    for f, df in tables.items():
        ajoute(f"Volume conserve - {f}", len(df) == volumes_avant[f],
               f"{volumes_avant[f]} -> {len(df)}")

    car = tables["caracteristiques"]
    ajoute("Num_Acc unique dans caracteristiques", car["Num_Acc"].is_unique)
    ajoute("(Num_Acc, id_vehicule) unique dans vehicules",
           not tables["vehicules"].duplicated(["Num_Acc", "id_vehicule"]).any())
    acc = set(car["Num_Acc"])
    for f in ["lieux", "vehicules", "usagers"]:
        orphelins = (~tables[f]["Num_Acc"].isin(acc)).sum()
        ajoute(f"Aucun orphelin Num_Acc - {f}", orphelins == 0, f"{orphelins} orphelins")
    veh = set(zip(tables["vehicules"]["Num_Acc"], tables["vehicules"]["id_vehicule"]))
    us = tables["usagers"]
    orph = sum((a, v) not in veh for a, v in zip(us["Num_Acc"], us["id_vehicule"]))
    ajoute("Usagers relies a un vehicule existant", orph == 0, f"{orph} orphelins")
    ajoute("Aucune valeur -1 dans les mesures numeriques",
           not any((tables["lieux"][c] == "-1").any() for c in ["vma_num", "nbv_num", "larrout_num"]))
    return pd.DataFrame(res)


def main():
    META.mkdir(parents=True, exist_ok=True)
    tables, volumes = {}, {}
    for f in FAMILLES:
        print(f"Nettoyage {f}...")
        df = lire(f)
        volumes[f] = len(df)
        df = r009_espaces(df, f)
        df = r010_marqueurs_vides(df, f)
        df = TRAITEMENTS[f](df)
        tables[f] = df

    controles = controler(tables, volumes)
    controles.to_csv(META / "controles_apres_nettoyage.csv", sep=";", index=False, encoding="utf-8-sig")
    pd.DataFrame(journal).to_csv(META / "journal_transformations.csv", sep=";", index=False, encoding="utf-8-sig")
    print(controles.to_string(index=False))

    if (controles["resultat"] == "ECHEC").any():
        sys.exit("\nCertains controles ont echoue : fichiers propres NON ecrits.")
    for f, df in tables.items():
        ecrire(df, f)
    print(f"\nFichiers propres ecrits dans {CLEAN}")


if __name__ == "__main__":
    main()


Writing scripts/nettoyer_baac.py


## 4. Exécution et contrôles d'intégrité

In [5]:
!python scripts/nettoyer_baac.py

Nettoyage caracteristiques...
Nettoyage lieux...
Nettoyage vehicules...
Nettoyage usagers...
                                    controle resultat           detail
          Volume conserve - caracteristiques       OK 327628 -> 327628
                     Volume conserve - lieux       OK 359512 -> 359512
                 Volume conserve - vehicules       OK 559847 -> 559847
                   Volume conserve - usagers       OK 745158 -> 745158
        Num_Acc unique dans caracteristiques       OK                 
(Num_Acc, id_vehicule) unique dans vehicules       OK                 
              Aucun orphelin Num_Acc - lieux       OK      0 orphelins
          Aucun orphelin Num_Acc - vehicules       OK      0 orphelins
            Aucun orphelin Num_Acc - usagers       OK      0 orphelins
       Usagers relies a un vehicule existant       OK      0 orphelins
Aucune valeur -1 dans les mesures numeriques       OK                 

Fichiers propres ecrits dans /content/P2-BAAC-2019-202

## 5. Résultats
### 5.1 Valeurs touchées par règle

In [6]:
j = pd.read_csv('metadata/nettoyage/journal_transformations.csv', sep=';', encoding='utf-8-sig')
j.groupby(['regle', 'famille', 'description'])['valeurs_touchees'].sum().reset_index()

,regle,famille,description,valeurs_touchees
0,R003,lieux,larrout : -1 ou non numerique -> vide,250710
1,R005,lieux,lartpc : -1 ou non numerique -> vide,0
2,R006,vehicules,occutc : -1 ou non numerique -> vide,0
3,R007,usagers,Doublon exact signale (conserve),164
4,R008,lieux,Lignes de lieux supplementaires conservees (rang_lieu > 1),31884
5,R009,caracteristiques,Espaces superflus supprimes,3733
6,R009,lieux,Espaces superflus supprimes,278599
7,R009,usagers,Espaces superflus supprimes,743384
8,R009,vehicules,Espaces superflus supprimes,2799
9,R010,caracteristiques,"Marqueurs N/A, null, #ERREUR remplaces par vide",5868


### 5.2 Détail par année

In [7]:
pd.pivot_table(j, index=['regle', 'description'], columns='annee', values='valeurs_touchees', aggfunc='sum', fill_value=0)

,annee,2019,2020,2021,2022,2023,2024
regle,description,,,,,,
R003,larrout : -1 ou non numerique -> vide,0,44214,53810,51456,52584,48646
R005,lartpc : -1 ou non numerique -> vide,0,0,0,0,0,0
R006,occutc : -1 ou non numerique -> vide,0,0,0,0,0,0
R007,Doublon exact signale (conserve),101,63,0,0,0,0
R008,Lignes de lieux supplementaires conservees (rang_lieu > 1),0,0,0,0,16038,15846
R009,Espaces superflus supprimes,138409,153344,185839,181524,187099,182300
R010,"Marqueurs N/A, null, #ERREUR remplaces par vide",427,45275,52181,51376,66120,66772
R012,Codes dep/com completes par des zeros a gauche,2966,0,0,0,0,0
R013,hrmn invalide -> flag (heure vide),0,0,0,0,0,0


### 5.3 Colonnes de signalement (flags) dans les fichiers propres

In [8]:
import glob, os
lignes = []
for f in sorted(glob.glob('data/clean/*.csv')):
    df = pd.read_csv(f, sep=';', dtype=str, encoding='utf-8-sig', keep_default_na=False)
    for c in [c for c in df.columns if c.startswith('flag_')]:
        n = (df[c] == '1').sum()
        lignes.append({'fichier': os.path.basename(f), 'flag': c, 'lignes_signalees': n,
                       'pourcentage': round(100 * n / len(df), 3)})
pd.DataFrame(lignes)

,fichier,flag,lignes_signalees,pourcentage
0,caracteristiques_2019_2024.csv,flag_heure_invalide,0,0.000
1,caracteristiques_2019_2024.csv,flag_date_invalide,0,0.000
2,caracteristiques_2019_2024.csv,flag_coord_hors_territoire,3115,0.951
3,lieux_2019_2024.csv,flag_nbv_num_hors_bornes,0,0.000
4,lieux_2019_2024.csv,flag_vma_num_hors_bornes,266,0.074
5,usagers_2019_2024.csv,flag_doublon_exact,164,0.022
6,usagers_2019_2024.csv,flag_age_hors_bornes,191,0.026


### 5.4 Contrôle de cohérence : nombre de tués par année
Ce total peut être comparé aux chiffres publiés par l'ONISR pour vérifier qu'aucune perte ne fausse les indicateurs.

In [9]:
u = pd.read_csv('data/clean/usagers_2019_2024.csv', sep=';', dtype=str, encoding='utf-8-sig', keep_default_na=False)
u[u['flag_doublon_exact'] == '0'].assign(est_tue=lambda d: d['est_tue'].astype(int)).groupby('annee')['est_tue'].sum()

,est_tue
annee,
2019,3497
2020,2780
2021,3219
2022,3550
2023,3398
2024,3432


## 6. Export des résultats

In [10]:
!zip -q -r /content/WP2_nettoyage_resultats.zip data/clean metadata/nettoyage scripts/nettoyer_baac.py
from google.colab import files
files.download('/content/WP2_nettoyage_resultats.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>